# quant_astro 解耦重构后的调用示例

对应这次重构后的 `core.py` + `kp.py` + `__init__.py`（不含 Dasha、不含庙旺落陷）。
按 5 个单元格走：安装/导入 → 配置 → 计算 → Dasha（可选）→ 打印全部原始结果。

这版在此基础上又做了三处完善：
- 安装时锁定 `pysweph>=2.10.3.3` 并打印实际装到的版本号，方便对照行为差异；
- `calculation_options` 里加了 `strict_ephe` 开关，星历文件还没配好时可以先设 `False` 跑通；
- 最后的打印环节终于用上了库里一直没人调用的 `decimal_to_dms`，黄经会同时给出十进制度和 DMS 字符串。

In [ ]:
# 先卸载旧的 pyswisseph（若存在）：pysweph 与 pyswisseph 共用同一个 import swisseph
# 命名空间，两者同时装在环境里会导致不确定用的是哪一个，这是 pysweph 官方迁移指南
# 建议的做法。如果你是在本地环境、已经装好了，这几行可以跳过不执行。
#
# 版本锁定 >=2.10.3.3：这份 core.py/kp.py 依赖该版本起 calc()/calc_ut() 新增的
# serr 错误信息，以及 houses 系列 cusps 数组从 2.10.3.4 起的新格式（两种格式代码
# 里都已经兼容），但至少要保证不会退回到更早、行为完全不同的版本——跟 setup.py
# 的 install_requires 保持一致。
!pip uninstall -y pyswisseph -q
!pip install -q "pysweph>=2.10.3.3"

# 如果这版 core.py/kp.py/__init__.py 是通过你自己的仓库或本地路径安装的，
# 把下面这行换成你自己的安装方式，例如：
# !pip install -e /path/to/your/quant_astro_project
# !pip install git+https://your-repo-url.git

import importlib.metadata
import swisseph as swe

print(f"pysweph 版本：{importlib.metadata.version('pysweph')}")

from quant_astro import (
    parse_time_geo,
    calculate_planets,
    calculate_minor_planets,
    calculate_fixed_stars,
    calculate_houses,
    solve_horary_houses,
    calculate_sunrise_sunset,
    calculate_day_lord,
    calculate_planetary_hour,
    get_kp_lords,
    kp_lookup,
    get_significators,
    get_ruling_planets,
    decimal_to_dms,
)

# Dasha（大运）模块这次解耦重构没有碰它，__init__.py 也没有重新导出它，
# 所以单独从子模块导入。如果你项目里这个文件名不一样，改一下这里的路径即可；
# 如果压根没有这个文件，下面会自动跳过第 4 个单元格。
try:
    from quant_astro.dasha_Vimshottari_api import create_dasha_table
except ImportError as exc:
    create_dasha_table = None
    print(f"⚠️ 没有找到 dasha_Vimshottari_api，Dasha 相关单元格会被跳过：{exc}")

In [ ]:
# ----------------- 设置所有占星参数 -----------------
chart_name = {"matter_name": "30 什么时候能找到工作"}

# 字段名和 parse_time_geo(**birth_config) 的参数名一一对应。
birth_config = {
    "local_time_str": "2026-08-20 20:47:00.000000",  # 本地时间（不带时区，两位小数秒）
    "timezone_str": "+8:00",                          # 时区（支持 +8、-5:30、+5.5）
    "latitude_str": "23°01′59.300000″",               # 纬度
    "longitude_str": "113°06′03.800000″",             # 经度
    "elevation": 0.0,     # 海拔（米）
    "atpress": 1013.25,   # 大气压
    "attemp": 20.0,       # 气温
    "calendar": "g",      # g = 格里历，j = 儒略历
}

calculation_options = {
    "ecliptic_mode": "tropical",           # 黄道模式：tropical / sidereal
    "ayanamsha_mode": "SIDM_KRISHNAMURTI", # 岁差体系（仅恒星黄道生效）
    "node_mode": "mean",                   # 交点类型：mean / true
    "house_system": "Regiomontanus",       # 宫位制，可用值见 quant_astro.HOUSE_SYSTEMS
    "heliocentric": False,                 # 日心制（原来的 USE_HELIOCENTRIC 这次真的接上了）

    # strict_ephe=True（默认）：本机缺 Swiss Ephemeris 高精度星历文件时直接报错，
    # 报错信息里会带上 Swiss Ephemeris 自己给的具体原因（比如缺了哪个 .se1 文件）。
    # 还没把 ephe 目录配置好、只是想先跑通流程看看结构时，可以先改成 False——
    # 会自动退回精度更低的 Moshier 算法并只发 RuntimeWarning，不会中断。
    # 正式产出数据前请务必确认这里是 True。
    "strict_ephe": True,

    # === 行星过滤 ===
    # 想要南北交点，'Ra'/'Ke' 要都显式写进来，不会因为写了 Ra 就自动带出 Ke。
    "selected_planets": ["Su", "Mo", "Me", "Ve", "Ma", "Ju", "Sa", "Ra", "Ke"],

    # 小行星现在是独立函数 calculate_minor_planets，这里留空表示不算；
    # 想算就填代码，例如 ["Ch", "Ce"]，可用值见 quant_astro.MINOR_PLANETS。
    "selected_minor_planets": [],

    # === 恒星配置 ===
    "selected_stars": [
        "Algol,bePer",       # 大陵五（英仙座β）
        "Alcyone,etTau",     # 昴宿六
        "Aldebaran,alTau",   # 毕宿五
        "Capella,alAur",     # 五车二
        "Sirius,alCMa",      # 天狼星
        "Procyon,alCMi",     # 南河三
        "Regulus,alLeo",     # 轩辕十四
        "Algorab,deCrv",     # 轸宿一
        "Spica,alVir",       # 角宿一
        "Arcturus,alBoo",    # 大角星
        "Alphecca,alCrB",    # 贯索四
        "Antares,alSco",     # 心宿二
        "Vega,alLyr",        # 织女一
        "DenebAlgedi,deCap", # 垒壁阵四
        "Fomalhaut,alPsA",   # 北落师门
    ],

    # kp卜卦：is_active=True 时用 solve_horary_houses 反推上升点，
    # False 时走普通的 calculate_houses。
    "KP_HORARY": {
        "is_active": False,
        "mode": "KS-N",  # KS-N(1-249) / CIL-N(1-2193)
        "number": 78,
    },
}

# rsmi 现在只表示"日出日落算法的样式位"（如 BIT_DISC_CENTER），
# CALC_RISE / CALC_SET 由 calculate_sunrise_sunset 内部按需自动补上，不用再传。
sunrise_config = {
    "rsmi": swe.BIT_DISC_CENTER,
}

# Dasha 配置，原样传给（未改动的）dasha_Vimshottari_api.create_dasha_table。
dasa_config = {
    "max_level": 2,          # 计算层级
    "output_mode": "all",    # "all" 包含所有层级，"present" 只含当前最小层级
    "days_in_year": 365.25,  # Dasha 计算中使用的年长度
}

In [ ]:
# ----------------- 执行计算 -----------------

# 1. 时间地理解析：后续所有函数共享的不可变上下文。
context = parse_time_geo(**birth_config)

# 2. 主行星 + 交点
planet_pos = calculate_planets(
    context,
    calculation_options["selected_planets"],
    ecliptic_mode=calculation_options["ecliptic_mode"],
    ayanamsha_mode=calculation_options["ayanamsha_mode"],
    node_mode=calculation_options["node_mode"],
    heliocentric=calculation_options["heliocentric"],
    strict_ephe=calculation_options["strict_ephe"],
)

# 3. 小行星（与主行星完全独立，按需单独调用）
minor_planet_pos = calculate_minor_planets(
    context,
    calculation_options["selected_minor_planets"],
    ecliptic_mode=calculation_options["ecliptic_mode"],
    ayanamsha_mode=calculation_options["ayanamsha_mode"],
    heliocentric=calculation_options["heliocentric"],
    strict_ephe=calculation_options["strict_ephe"],
)

# 4. 恒星（同样独立；只需要 jd_utc，直接传 context 也可以）
fixed_star_pos = calculate_fixed_stars(
    context,
    calculation_options["selected_stars"],
    ecliptic_mode=calculation_options["ecliptic_mode"],
    ayanamsha_mode=calculation_options["ayanamsha_mode"],
    strict_ephe=calculation_options["strict_ephe"],
)

# 5. 宫位：普通命盘用 calculate_houses；卜卦(is_active=True)用 solve_horary_houses。
#    两者返回的字典都带 houses / axes / auxiliary_points，下面的代码不用区分。
#    注：宫位计算本身不读取行星星历文件（纯球面三角），不受 strict_ephe 影响，
#    所以这两个函数都没有这个参数。
_horary = calculation_options["KP_HORARY"]
if _horary.get("is_active"):
    house_result = solve_horary_houses(
        context,
        _horary["number"],
        mode=_horary.get("mode", "KS-N"),
        house_system=calculation_options["house_system"],
        ecliptic_mode=calculation_options["ecliptic_mode"],
        ayanamsha_mode=calculation_options["ayanamsha_mode"],
    )
else:
    house_result = calculate_houses(
        context,
        calculation_options["house_system"],
        ecliptic_mode=calculation_options["ecliptic_mode"],
        ayanamsha_mode=calculation_options["ayanamsha_mode"],
    )

house_pos = house_result["houses"]                    # "house 1".."house 12"
axes = house_result["axes"]                           # Asc / Desc / MC / IC
auxiliary_points = house_result["auxiliary_points"]   # ARMC / Vertex / ...

# 6. 日出日落 / 值日星 / 行星时——三者共用同一套日出日落算法。
sun_events = calculate_sunrise_sunset(context, rsmi=sunrise_config["rsmi"])
day_lord_result = calculate_day_lord(context, sun_events=sun_events)
planetary_hour_data = calculate_planetary_hour(context, rsmi=sunrise_config["rsmi"])

# ----------------- KP计算 -----------------

# 1. 获取KP星主：house_pos 是扁平的 "house N" -> {'lon':...} 字典，
#    get_kp_lords/get_significators 也都能直接吃 house_result 整个字典（会自动解包）。
kp_planet_results, kp_house_results = get_kp_lords(planet_pos, house_pos)

# 2. 象征星（ABCD / 1234），内部只建一次共享映射。
kp_planet_sigs, kp_house_sigs = get_significators(
    planet_pos, house_pos, kp_planet_results, kp_house_results
)

# 3. 主宰星：真实 Asc 用 axes['Asc']['lon'] 单独查一次 KP 星主再传进去——
#    Whole Sign / Vehlow 等宫位制下，第一宫宫头并不等于真实上升点，
#    不能再用 kp_house_results['house 1'] 代替。
asc_kp_result = kp_lookup(axes["Asc"]["lon"])
kp_ruling_planets = get_ruling_planets(
    kp_planet_results, kp_house_results, day_lord_result, asc_kp_result=asc_kp_result,
)

In [ ]:
# ----------------- Dasha计算 -----------------
# dasha_Vimshottari_api 没有被这次重构改动，这里假设它的调用方式跟以前一样：
# create_dasha_table(planet_pos, birth_config, dasa_config)。
# 如果它的签名其实变了，或者需要 planet_pos 里有这版 calculate_planets 已经
# 不再返回的 'dec_speed' 字段，请对照那个文件自行调整这里的调用。
if create_dasha_table is not None:
    print("\n--- 开始生成Dasha时间表 ---")
    try:
        create_dasha_table(planet_pos, birth_config, dasa_config)
    except Exception as exc:
        print(f"⚠️ Dasha 计算失败，请检查 dasha_Vimshottari_api 是否兼容新的 planet_pos 结构：{exc}")
else:
    print("（未找到 dasha_Vimshottari_api，跳过 Dasha 计算）")

In [ ]:
# ----------------- 检查所有原始结果词典 -----------------


def _format_lon(entry):
    """在原始字典后面附一个 DMS 格式的黄经，方便肉眼读数。

    注意 auxiliary_points 里的 ARMC 是 {'angle': ..., 'speed': ...} 结构——
    它是赤道坐标角，不是黄经，字段名跟其余轴点的 'lon' 不一样，所以这里两个
    键都试一下，两个都没有就返回空字符串（不会因为结构不统一而 KeyError）。
    """
    lon = entry.get("lon", entry.get("angle"))
    if lon is None:
        return ""
    return decimal_to_dms(lon)["str"]


print("\n" + "="*20 + " 时间地理上下文 (context) " + "="*20)
print(context.to_dict())

print("\n" + "="*20 + " 行星原始位置字典 (planet_pos) " + "="*20)
for key, value in planet_pos.items():
    print(f"{key}: {value}  |  黄经 {_format_lon(value)}")

print("\n" + "="*20 + " 小行星原始位置字典 (minor_planet_pos) " + "="*20)
for key, value in minor_planet_pos.items():
    print(f"{key}: {value}  |  黄经 {_format_lon(value)}")

print("\n" + "="*20 + " 恒星原始位置字典 (fixed_star_pos) " + "="*20)
for key, value in fixed_star_pos.items():
    print(f"{key}: {value}  |  黄经 {_format_lon(value)}")

print("\n" + "="*20 + " 宫头 (house_pos) " + "="*20)
for key, value in house_pos.items():
    print(f"{key}: {value}  |  黄经 {_format_lon(value)}")

print("\n" + "="*20 + " 四轴 (axes: Asc/Desc/MC/IC) " + "="*20)
for key, value in axes.items():
    print(f"{key}: {value}  |  黄经 {_format_lon(value)}")

print("\n" + "="*20 + " 辅助轴点 (auxiliary_points) " + "="*20)
for key, value in auxiliary_points.items():
    print(f"{key}: {value}  |  {_format_lon(value)}")

# ── KP ──

print("\n" + "="*20 + " KP行星结果字典 (kp_planet_results) " + "="*20)
for key, value in kp_planet_results.items():
    print(f"{key}: {value}")

print("\n" + "="*20 + " KP宫位结果字典 (kp_house_results) " + "="*20)
for key, value in kp_house_results.items():
    print(f"{key}: {value}")

print("\n" + "="*20 + " 真实 Asc 的 KP 星主 (asc_kp_result) " + "="*20)
print(asc_kp_result)

print("\n" + "="*20 + " 行星象征宫位 ABCD (kp_planet_sigs) " + "="*20)
for key, value in kp_planet_sigs.items():
    print(f"{key}: {value}")

print("\n" + "="*20 + " 宫位象征星 1234 (kp_house_sigs) " + "="*20)
for key, value in kp_house_sigs.items():
    print(f"{key}: {value}")

print("\n" + "="*20 + " 主宰星字典 (kp_ruling_planets) " + "="*20)
print(kp_ruling_planets)

# ── 其他 ──

print("\n" + "="*20 + " 日出日落 (sun_events) " + "="*20)
print(sun_events)

print("\n" + "="*20 + " 值日星 (day_lord_result) " + "="*20)
print(day_lord_result)

print("\n" + "="*20 + " 行星时数据 (planetary_hour_data) " + "="*20)
for key, value in planetary_hour_data.items():
    print(f"{key}: {value}")

print("\n" + "="*20 + " 儒略日 (jd_utc) " + "="*20)
print(context.jd_utc)

if calculation_options["KP_HORARY"].get("is_active"):
    print("\n" + "="*20 + " 卜卦反推额外信息 " + "="*20)
    print({
        k: house_result[k]
        for k in ("number", "mode", "target_asc", "solved_jd_utc", "solved_local_dt")
    })